## Märgendatud nimisõnafraasid ja verbifraasid


In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import numpy as np
import copy
import os
import configparser
import json
import time
import csv

In [2]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

In [3]:
RESULT_DIR = "../../results/"

DATA_FILE1 = RESULT_DIR+ "n80_examples_large_v01/gpt_v03/"+ "gpt_10K_b10_run01.csv"

DATA_FILE2 = RESULT_DIR+ "n20_examples_large_v01/gpt_v02/" + "gpt_b10_run01.csv"

DATA_FILE3 = RESULT_DIR+ "n50_examples_large_v01/gpt_v02/"+ "gpt_b10_run01.csv"

SUMMARY_DIR = "../summary/"

SUMMARY_FILE = SUMMARY_DIR+ "verbi_nimisonafraasid.csv"


## Vastustega df

In [24]:
df1 = pd.read_csv(DATA_FILE1, encoding="utf-8", sep=",").fillna("")
df2 = pd.read_csv(DATA_FILE2, encoding="utf-8", sep=",").fillna("")
df3 = pd.read_csv(DATA_FILE3, encoding="utf-8", sep=",").fillna("")

In [49]:
cols = ['sentence_id', 'head_id', 'head_loc', 'verb', 'verb_compound','morph_case', 'lemma', 'form']

combined = pd.concat([df1[cols], df2[cols]])
combined = pd.concat([combined, df3[cols]])

In [26]:
combined

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form
0,2792027,4477403,1,sadama,maha,in,Tallinn,Tallinnas
1,2149636,3433963,8,ringlema,,in,piletiäri,piletiäris
2,12372585,19808269,6,sööma,,in,fuajee,fuajees
3,6427967,10334047,1,kiirustama,,adit,õnnetuspaik,Õnnetuspaika
4,13574774,21705880,5,lubama,,ill,Vilnius,Vilniusesse
...,...,...,...,...,...,...,...,...
9995,981021,1562806,8,levima,,ad,selgitus,selgitusel
9996,15749627,24570642,10,saatma,,el,laevastik,laevastikust
9997,974771,1552554,7,tõttama,,all,kolleeg,kolleegidele
9998,4825664,7748191,4,hüppama,,el,loodetud,loodetust


In [27]:
combined.groupby(['verb','verb_compound'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,verb,verb_compound,count
618,saama,,209
178,kaduma,,171
170,jõudma,,168
367,lubama,,161
152,jääma,,160
...,...,...,...
364,loosima,välja,1
507,pakkima,,1
742,testima,,1
872,viima,läbi,1


In [28]:
combined.groupby(['verb','verb_compound', 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,verb,verb_compound,morph_case,count
1063,suunduma,,el,72
291,kaduma,,ill,71
229,jälgima,,in,71
137,hakkama,,ill,69
1186,tervitama,,in,69
...,...,...,...,...
849,petma,,in,1
581,loosima,välja,ad,1
817,pakkima,,ad,1
1446,võõrandama,,all,1


In [29]:
combined.groupby(['form'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,form,count
6280,ajal,39
1560,Eestis,35
5423,Tallinnas,34
6651,alusel,30
4730,Pärnus,27
...,...,...
7505,avastusele,1
7506,avastusest,1
7507,avastusse,1
7508,avastustel,1


In [30]:
combined.groupby(['lemma'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,lemma,count
1043,Eesti,89
3188,Tallinn,72
3647,aeg,71
2379,Moskva,61
10034,mis,60
...,...,...
7207,kaljutunnel,1
7208,kaljutuvi,1
7209,kalkulatsioon,1
7210,kalkuleerimine,1


In [31]:
combined.groupby(['lemma', 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,lemma,morph_case,count
4800,aeg,ad,41
1339,Eesti,in,35
4188,Tallinn,in,34
9153,juht,ad,29
11335,kord,ad,28
...,...,...,...
7157,enamjuht,ad,1
7159,enampakkumine,all,1
7160,enamus,ad,1
7161,enamus,all,1


In [32]:
combined.groupby(['form', 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,form,morph_case,count
6280,ajal,ad,39
1560,Eestis,in,35
5423,Tallinnas,in,34
6653,alusel,ad,30
4730,Pärnus,in,27
...,...,...,...
7508,avastusele,all,1
7509,avastusest,el,1
7510,avastusse,adit,1
7511,avastustel,ad,1


In [50]:
result = (
    combined.groupby("form")["morph_case"]
      .agg(
          case_count="nunique",        # how many different cases
          cases=lambda x: sorted(x.unique())  # which cases
      )
      .reset_index()
)
result

,form,case_count,cases
0,.-80ndatesse,1,[ill]
1,.G-le,1,[all]
2,/a-le,1,[all]
3,"0 , 5-le",1,[all]
4,"0,5 -st",1,[el]
...,...,...,...
20273,šokist,1,[el]
20274,šokolaadikohvikusse,1,[ill]
20275,šotlasest,1,[el]
20276,žalgiris,1,[in]


In [51]:
result[result["case_count"]>1]

,form,case_count,cases
6334,ajas,3,"[ad, all, in]"
7092,arvutisse,2,"[adit, ill]"
8183,eestisse,2,"[adit, ill]"
9663,hoidlasse,2,"[adit, ill]"
15238,mõrvas,2,"[ad, in]"
16037,pardaarvutisse,2,"[adit, ill]"
19079,tundmatusse,2,"[adit, ill]"
19724,viis,2,"[ad, ill]"


In [59]:
combined[(combined["form"]=="arvutisse") & (combined["morph_case"]=="ill")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form
1015,9683051,15538849,20,edastama,,ill,arvuti,arvutisse
3745,5612308,8999098,9,lubama,,ill,arvuti,arvutisse
9765,346810,535906,6,kirjutama,,ill,arvuti,arvutisse


In [33]:
DATA_DIR = "../../data/"

DATA_FILE11 = DATA_DIR+ "n80_examples_large_v02.csv"

DATA_FILE12 = DATA_DIR+ "n20_examples_large_v01.csv"

DATA_FILE13 = DATA_DIR+ "n50_examples_large_v01.csv"


In [34]:
df11 = pd.read_csv(DATA_FILE11, encoding="utf-8", sep=",").fillna("")
df12 = pd.read_csv(DATA_FILE12, encoding="utf-8", sep=",").fillna("")
df13 = pd.read_csv(DATA_FILE13, encoding="utf-8", sep=",").fillna("")

In [41]:
cols = ['sentence_id', 'head_id', 'head_loc', 'verb', 'verb_compound','morph_case', 'lemma', 'form']

combined2 = pd.concat([df11[cols], df12[cols]])
combined2 = pd.concat([combined2, df13[cols]])

In [42]:
combined2

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form
0,3143873,5047902,12,olema,vaja,ill,eelarve,eelarvesse
1,5356377,8592242,12,kaevama,,in,eluaasta,eluaastates
2,11180684,17909605,6,peitma,,ill,maa,maasse
3,668202,1062267,4,ringlema,,in,Eesti,Eestis
4,1325496,2107081,9,treenima,,in,Ramsau,Ramsaus
...,...,...,...,...,...,...,...,...
179834,16527876,25486831,3,saabuma,,all,juhataja,juhatajale
179835,3784092,6094852,3,peksma,,ad,pööripäev,pööripäeval
179836,19698,33955,8,andma,,adit,käsutus,käsutusse
179837,8901024,14307437,23,ajama,,all,naine,naistele


In [43]:
combined2.groupby(['verb','verb_compound'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,verb,verb_compound,count
621,saama,,3001
170,jõudma,,3000
152,jääma,,2500
752,tooma,,2479
742,tekkima,,2136
...,...,...,...
311,külmutama,,17
413,mattuma,,17
715,tagurdama,otsa,17
424,minema,järele,16


In [44]:
combined2.groupby(['verb','verb_compound', 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,verb,verb_compound,morph_case,count
654,maksma,,in,501
988,saama,,adit,501
789,omama,,in,501
1080,säilima,,in,501
802,ostma,,el,501
...,...,...,...,...
501,külmutama,,ad,17
1292,tõstma,välja,ad,17
660,mattuma,,ad,17
678,minema,järele,ad,16


In [45]:
combined2.groupby(['form'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,form,count
46426,alusel,388
44180,ajal,374
123205,sõnul,345
71840,juhul,286
10653,Eestis,276
...,...,...
53302,dalavahetus,1
53300,dalai-laamal,1
53299,daisidest,1
53298,daiquiri,1


In [46]:
combined2.groupby(['lemma'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,lemma,count
6028,Eesti,753
21951,aeg,722
56084,mis,706
55981,mina,681
19057,Tallinn,628
...,...,...
44183,kiviaste,1
44184,kivihoone,1
16906,Renteri,1
44186,kivijaht,1


In [47]:
combined2.groupby(['lemma', 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,lemma,morph_case,count
31738,aeg,ad,392
108209,sõna,ad,345
58611,juht,ad,341
35213,andmed,ad,335
30975,aasta,ad,322
...,...,...,...
47686,esinemissagedus,in,1
47685,esinemissagedus,el,1
47684,esinemissagedus,all,1
47682,esinemisruum,el,1


In [48]:
combined2.groupby(['form', 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,form,morph_case,count
46447,alusel,ad,388
44196,ajal,ad,374
123353,sõnul,ad,345
71882,juhul,ad,286
10656,Eestis,in,276
...,...,...,...
53367,debüütalal,ad,1
53363,debütantidest,el,1
53362,debütantidele,all,1
53359,debütandil,ad,1
